# G×S grasp detection — full pipeline on Colab
Runs data preparation, frozen-CLIP feature caching, training of every component and evaluation. State is kept on Google Drive (`DRIVE`), so after a disconnect just run all cells again: finished steps are skipped.

In [ ]:
REPO = 'https://github.com/RyleHan/gxs-grasp.git'   # set to the repository URL
DRIVE = '/content/drive/MyDrive/gxs'
G_EPOCHS, S_EPOCHS = 12, 4
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
import os
if not os.path.exists('/content/gxs-grasp'):
    !git clone -q $REPO /content/gxs-grasp
else:
    !git -C /content/gxs-grasp pull -q
%cd /content/gxs-grasp
!pip -q install hf_transfer
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

## 1. Data (≈20 min the first time, then restored from Drive)

In [ ]:
import os
tar = f'{DRIVE}/data.tar'
if os.path.exists(tar) and not os.path.exists('/content/data/annotations.pkl'):
    !tar -xf $tar -C /content
    !cp -r {DRIVE}/splits/* splits/ 2>/dev/null || true
if not os.path.exists('/content/data/annotations.pkl'):
    !python scripts/setup_data.py --raw /content/raw --out /content/data --cache /content/cache
    !tar -cf $tar -C /content data
    !mkdir -p {DRIVE}/splits && cp splits/*.txt splits/stats.json {DRIVE}/splits/
!cat splits/stats.json | head -60

## 2. Frozen CLIP features (local disk, recomputed per session, ≈10 min)

In [ ]:
import os
if not os.path.exists('/content/clip/clip_index.json'):
    !python scripts/precompute_clip.py --data /content/data --out /content/clip --batch 32
!ls -la /content/clip

## 3. Train all components and evaluate every row

In [ ]:
!WORKERS=2 bash scripts/run_all.sh /content/data /content/clip {DRIVE}/runs {G_EPOCHS} {S_EPOCHS}

In [ ]:
!cat {DRIVE}/runs/results/table.md

## 4. Diagnostics and figure
Upper bound with perfect selection, a single- vs multi-object breakdown, and the qualitative figure (examples picked automatically among test scenes where G×S gets both objects right and G alone does not).

In [ ]:
for s in ['test_seen', 'test_unseen']:
    !python scripts/evaluate.py --method oracle --split {s} --data /content/data --clip-dir /content/clip --g {DRIVE}/runs/g/best.pt --out {DRIVE}/runs/results/oracle_{s}.json --workers 2 | tail -n 12
!python scripts/make_table.py {DRIVE}/runs/results
!python scripts/analyse.py {DRIVE}/runs/results --split test_seen

In [ ]:
# paste two '--scenes ... --objs a,b' pairs printed above
SCENES, OBJS = 'SCENE_A,SCENE_B', '0,1;0,1'
!python scripts/visualize.py --data /content/data --clip-dir /content/clip --g {DRIVE}/runs/g/best.pt --s {DRIVE}/runs/s/best.pt --scenes {SCENES} --objs '{OBJS}' --out {DRIVE}/fig_qual.png
from IPython.display import Image
Image(f'{DRIVE}/fig_qual.png')